# VELALAR COLLEGE OF ENGINEERING AND TECHNOLOGY, ERODE - 12
## DEPARTMENT OF INFORMATION TECHNOLOGY
### Course: 25ITC31 Foundations of Data Science | Academic Year: 2026-27 (ODD)
---
## Project Assessment: Scenario 14
### Electricity Consumption Management and Energy Usage Analysis System
**Student Register Numbers:** 44 (Kanishka - Class 25IT3A)

This notebook demonstrates:
1. Data Ingestion & SQL Database extraction
2. Data Cleaning, Type Casting & Preprocessing
3. Outlier Detection using the IQR Method
4. Descriptive Statistics & Sector-wise Aggregations
5. Exploratory Data Analysis (EDA) & Visualizations with Matplotlib/Seaborn

### 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set visual style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 150

### 2. Load Dataset from Database / CSV

In [2]:
dataset_path = '../database/electricity_consumption_dataset.csv'
df = pd.read_csv(dataset_path)
print(f"Loaded {len(df)} records with {df.shape[1]} attributes.")
df.head()

### 3. Data Preprocessing & Missing Values Check

In [3]:
# Check data types and nulls
print("Missing values summary:")
print(df.isnull().sum())

# Date conversions
df['reading_date'] = pd.to_datetime(df['reading_date'])
df['bill_date'] = pd.to_datetime(df['bill_date'])
df['due_date'] = pd.to_datetime(df['due_date'])

# Create period label
df['period_label'] = df['billing_year'].astype(str) + '-' + df['billing_month'].astype(str).str.zfill(2)
df.info()

### 4. Outlier Detection (IQR Method for Domestic Sector)

In [4]:
domestic_df = df[df['connection_type'] == 'Domestic']
q1 = domestic_df['units_consumed'].quantile(0.25)
q3 = domestic_df['units_consumed'].quantile(0.75)
iqr = q3 - q1
upper_threshold = q3 + (1.5 * iqr)

print(f"Q1: {q1:.1f}, Q3: {q3:.1f}, IQR: {iqr:.1f}")
print(f"Upper Outlier Threshold for Domestic: {upper_threshold:.1f} kWh")
anomalous_users = domestic_df[domestic_df['units_consumed'] > upper_threshold]
anomalous_users[['consumer_id', 'full_name', 'period_label', 'units_consumed', 'total_amount']]

### 5. Descriptive Statistics by Connection Type

In [5]:
summary = df.groupby('connection_type')['units_consumed'].agg(['count', 'mean', 'std', 'median', 'min', 'max'])
summary.round(2)

### 6. Visualization 1: Monthly Consumption Trend (Line Chart)

In [6]:
plt.figure(figsize=(10, 5.5))
trend = df.groupby(['period_label', 'connection_type'])['units_consumed'].sum().unstack()
for col in trend.columns:
    plt.plot(trend.index, trend[col], marker='o', linewidth=2.5, label=col)
plt.title('Monthly Electricity Consumption Trend Across Categories (May-Oct 2026)', fontsize=13, fontweight='bold')
plt.xlabel('Billing Period', fontweight='bold')
plt.ylabel('Total Units Consumed (kWh) - Log Scale', fontweight='bold')
plt.yscale('log')
plt.legend(title='Category')
plt.tight_layout()
plt.show()

### 7. Visualization 2: Top 10 High Energy Consumers (Horizontal Bar Chart)

In [7]:
plt.figure(figsize=(9, 5.5))
oct_data = df[df['period_label'] == '2026-10'].sort_values(by='units_consumed', ascending=True).tail(10)
plt.barh(oct_data['full_name'], oct_data['units_consumed'], color='#34495e', edgecolor='black')
plt.title('Top 10 High Energy Consumers (October 2026)', fontsize=13, fontweight='bold')
plt.xlabel('Units Consumed (kWh)', fontweight='bold')
plt.ylabel('Consumer Name', fontweight='bold')
plt.tight_layout()
plt.show()

### 8. Visualization 3: Revenue Distribution by Sector (Pie / Donut Chart)

In [8]:
plt.figure(figsize=(7, 7))
revenue = df.groupby('connection_type')['total_amount'].sum()
plt.pie(revenue, labels=revenue.index, autopct='%1.1f%%', startangle=140,
        colors=['#f39c12', '#2980b9', '#8e44ad', '#27ae60'],
        wedgeprops=dict(width=0.6, edgecolor='white', linewidth=2))
plt.title('Total Billed Revenue Contribution by Sector', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 9. Visualization 4: Correlation Heatmap

In [9]:
plt.figure(figsize=(7, 5.5))
corr_features = ['sanctioned_load_kw', 'previous_reading', 'current_reading', 'units_consumed', 'total_amount']
sns.heatmap(df[corr_features].corr(), annot=True, cmap='Blues', fmt='.2f', linewidths=1)
plt.title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()